# 16. Embedding 실험 집계 및 Figure

notebook 14, 15의 결과를 합쳐 Figure 16~19와 요약 CSV를 만듭니다.

| Figure | 내용 |
|---|---|
| 16 | timeout 대 embedding 성공률 (topology 비교, instance별 panel) |
| 17 | 성공한 embedding의 품질 (physical qubits, avg/max chain length) |
| 18 | tries 대 embedding 성공률 (timeout 고정) |
| 19 | QUBO 엣지 수와 성공률의 관계 |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
STUDY = config["embedding_study"]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


## 결과 로드

`embedding_study.csv`는 notebook 14, 15의 결과가 **누적**된 파일입니다. 이전에 다른 조건으로 돌린 행이 남아 있을 수 있으므로, 무엇이 들어 있는지 먼저 확인하십시오.

특정 topology만 보고 싶으면 `TOPOLOGY_FILTER`를 지정하십시오. `None`이면 전부 사용합니다.

In [ ]:
from src.persistence import load_table, save_table
from src import embedding_study as ES
from src import embedding_plotting as EP

TOPOLOGY_FILTER = None   # 예: ["zephyr"] 또는 ["pegasus", "zephyr"]

results = load_table(PROCESSED_DIR, "embedding_study.csv")
print("파일에 들어 있는 조건:")
print(
    results.groupby(["sweep", "topology", "instance"])["success"]
    .agg(["count", "mean"])
)

if TOPOLOGY_FILTER is not None:
    before = len(results)
    results = results[results["topology"].isin(TOPOLOGY_FILTER)]
    print(f"\n필터 적용: {before} -> {len(results)} 행")
results.head()

## 요약 CSV

`min_timeout_success`는 embedding이 처음 성공한 최소 timeout입니다. 값이 있으면 **search budget으로 넘을 수 있는 벽**이었다는 뜻이고, `NaN`이면 모든 timeout에서 실패했다는 뜻입니다.

In [ ]:
summary = ES.summarize(results)
save_table(summary, PROCESSED_DIR, "embedding_study_summary.csv")
print("저장:", PROCESSED_DIR / "embedding_study_summary.csv")
summary

## Figure 16 — timeout 대 성공률

In [ ]:
from IPython.display import Image

path = EP.plot_timeout_success(results, FIGURE_DIR)
Image(filename=str(path))

## Figure 17 — 성공한 embedding의 품질

성공했더라도 chain이 길면 chain break 확률이 높아 실제 solution quality가 나빠집니다. **성공/실패만이 아니라 품질도 함께 봐야** topology의 이점을 정확히 평가할 수 있습니다.

In [ ]:
path = EP.plot_embedding_quality(results, FIGURE_DIR)
Image(filename=str(path))

## Figure 18 — tries 대 성공률

In [ ]:
path = EP.plot_tries_success(results, FIGURE_DIR)
Image(filename=str(path))

## Figure 19 — 그래프 규모와 성공률

가로축이 변수 수가 아니라 **이차항 수**입니다. 점의 크기가 변수 수를 나타냅니다. 변수가 적은데도 실패하는 점이 있다면, 성공을 가르는 것이 변수 수가 아니라 그래프의 조밀도라는 뜻입니다.

In [ ]:
path = EP.plot_density_vs_success(results, FIGURE_DIR)
Image(filename=str(path))

## 원인 판정

instance별로 세 원인 중 어느 것인지 자동 판정합니다. 자동 판정은 보조 지표이므로, 위 그림과 요약표를 함께 보고 최종 판단하십시오.

In [ ]:
lines = ES.diagnose(results)
for line in lines:
    print(line)

summary_path = PROCESSED_DIR / "embedding_study_summary.txt"
summary_path.write_text("\n".join(lines), encoding="utf-8")
print("\n저장:", summary_path)

In [ ]:
import requests

NTFY_API_KEY = "cflp-formulation-260910"

def notify(message):
    requests.post(
        f"https://ntfy.sh/{NTFY_API_KEY}",
        data=message.encode("utf-8")
    )